# Dataset de entrenamiento — cross-encoder: ampliación con sentencias SL (Casación Laboral)

Cuarta vía para ampliar el corpus. Las tres anteriores (`ds_parte1_redal.ipynb`,
`ds_parte2_corte_const.ipynb`, `ds_parte3_research_list.ipynb`) sacan todo de tutelas T/SU de
la Corte Constitucional, y eso dejó sin cobertura tres bloques centrales del CST — arts.
127-132 (factores salariales), 158-176 (jornada/horas extras), 186-192 (vacaciones) — porque
esas disputas casi nunca llegan a tutela (improcedentes por subsidiariedad) y se litigan por
vía ordinaria hasta casación. Este notebook trae sentencias **SL de la Sala de Casación
Laboral de la Corte Suprema de Justicia**, que sí resuelven esos temas directamente.

**Fuente:** `cortesuprema.gov.co/sala-de-casacion-laboral-relatoria-reiteraciones-relevantes/`
— página estática (sin auth, sin JS) con "reiteraciones relevantes" organizadas por año
(2017-2024) y por materia, entre ellas la pestaña **Laboral Individual** que ya viene
pre-filtrada por tema. Es un conjunto chico (26 entradas en total) pero de alta precisión —
no requiere el embudo de keywords + gate que usan los otros notebooks, aunque igual se aplica
el filtro de pertinencia por consistencia y para atrapar los casos que resulten ser en
realidad de sector público o seguridad social.

**Verificado manualmente antes de automatizar (ver detalle en la sección 2):** el identificador
visible en la página (ej. `SL3100-2024`) no siempre coincide con el año real del archivo PDF
(ej. `SL3100-2023.pdf`) — no se puede construir la URL del PDF por patrón, hay que resolver
cada caso vía su página intermedia (`cortesuprema.gov.co/rl_dl_sl{id}`), donde el PDF está
embebido en un `<iframe>`, no en un `<a href>`.

**Nota importante sobre falsos negativos** (aportada por el research que mapeó esta fuente):
una EPS o AFP como **empleadora demandada** en un pleito de salario o despido SÍ es laboral
individual válido, no seguridad social — hay que mirar en qué calidad aparece la entidad, no
descartar solo por ver "EPS"/"AFP"/"Colpensiones" en el texto.

Comparte la misma blacklist (`dataset_cross_encoder.csv` / `descartados.csv`) que los otros
tres notebooks.

**Contenido:**
1. Setup
2. Extracción de candidatas (pestaña "Laboral Individual", 2017-2024)
3. Resolución de PDF y descarga de texto
4. Prefiltro de keywords
5a. Filtro barato de pertinencia (Haiku)
5b. Extracción estructurada completa (Sonnet)
6. Construcción de pares
7. Loop principal con checkpointing
8. Resultado acumulado

## 1. Setup

In [1]:
# En Colab: descomenta la siguiente línea (o usa `pip install -r requirements.txt` en el venv local)
# !pip install -q beautifulsoup4 requests pandas anthropic python-dotenv pdfplumber

import os
import re
import time
import json
from io import BytesIO

import requests
import pandas as pd
import pdfplumber
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

HEADERS = {"User-Agent": "Mozilla/5.0"}

# Mismas rutas acumuladas que los otros tres notebooks — blacklist compartida
PARES_PATH = "../../data/dataset_cross_encoder.csv"
DESCARTES_PATH = "../../data/descartados.csv"
COLUMNAS_PARES = ["consulta", "articulo", "tipo", "label", "sentencia_origen"]
COLUMNAS_DESCARTES = ["sentencia", "razon"]

def cargar_si_existe(path, columnas):
    if os.path.exists(path):
        return pd.read_csv(path)
    return pd.DataFrame(columns=columnas)

print("Setup listo.")

Setup listo.


## 2. Extracción de candidatas (pestaña "Laboral Individual", 2017-2024)

La página tiene una pestaña por año (2017-2024), cada una con sub-pestañas por materia
(Pensiones, Laboral Individual, Laboral Colectivo, Procedimiento Laboral, Seguridad Social,
Recurso de Casación, Riesgos). Es una sola página HTML estática — las 8 secciones "Laboral
Individual" (una por año) están todas en el mismo documento, ocultas por CSS/JS del lado del
cliente, así que un simple `requests.get` + BeautifulSoup las trae todas sin necesidad de
navegador headless. Cada entrada trae el identificador (texto del link, ej. `SL3100-2024`),
una descripción temática, y un link a una página intermedia.

In [2]:
URL_REITERACIONES = "https://cortesuprema.gov.co/sala-de-casacion-laboral-relatoria-reiteraciones-relevantes/"

resp = requests.get(URL_REITERACIONES, headers=HEADERS, timeout=30)
resp.raise_for_status()
soup = BeautifulSoup(resp.content, "html.parser")

divs = soup.find_all("div", id="laboral-individual-tab")
print(f"Bloques 'Laboral Individual' encontrados (uno por año): {len(divs)}")

candidatas = []
for div in divs:
    for li in div.find_all("li"):
        a = li.find("a")
        p = li.find("p")
        if a and a.get("href"):
            candidatas.append({
                "identificador": a.get_text(strip=True),
                "tema": p.get_text(strip=True) if p else "",
                "url_intermedia": a["href"],
            })

df_candidatas_sl = pd.DataFrame(candidatas).drop_duplicates(subset="identificador").reset_index(drop=True)
print(f"Candidatas 'Laboral Individual' (todas las años, sin duplicar): {len(df_candidatas_sl)}")
df_candidatas_sl[["identificador", "tema"]]

Bloques 'Laboral Individual' encontrados (uno por año): 8
Candidatas 'Laboral Individual' (todas las años, sin duplicar): 39


,identificador,tema
0,SL3100-2024,Si bien el reconocimiento de la pensión de vej...
1,SL863-2024,Aunque el trabajador acuerde una “suspensión d...
2,SL1392-2024,"Se protege al trabajador, cuando el contrato i..."
3,SL1833-2023,Decreto de pruebas de oficio: entre la faculta...
4,SL1514-2023,El simple sometimiento del asalariado a dispon...
5,SL1166-2023,La configuración de la calidad de padre cabeza...
6,SL1050-2023,La aplicación del enfoque de género no vulnera...
7,SL2338-2023,
8,SL020-2023,
9,SL464-2023,


### 2b. Candidatas adicionales — boletines históricos (research 5)

Segunda fuente de candidatas SL, sacada de boletines trimestrales de la Sala (research
paralelo con Claude web) en vez de "reiteraciones relevantes". A diferencia de las anteriores,
estas ya traen su URL de **PDF completo verificada** (`textos_completos_verificados.csv`, GET
200 confirmado antes de correr este notebook) — se saltan el paso de resolución vía página
intermedia.

In [3]:
RUTA_TEXTOS_VERIFICADOS = "../../textos_completos_verificados.csv"

df_candidatas_sl["url_pdf_directo"] = None

if os.path.exists(RUTA_TEXTOS_VERIFICADOS):
    df_verificados = pd.read_csv(RUTA_TEXTOS_VERIFICADOS)
    df_verificados = df_verificados[df_verificados["verificado"] == "si"]
    df_extra = pd.DataFrame({
        "identificador": df_verificados["sentencia"],
        "tema": "boletin_historico_research5",
        "url_intermedia": None,
        "url_pdf_directo": df_verificados["url_pdf_completo"],
    })
    df_candidatas_sl = pd.concat([df_candidatas_sl, df_extra], ignore_index=True) \
        .drop_duplicates(subset="identificador").reset_index(drop=True)
    print(f"Candidatas agregadas desde research 5 (texto completo ya resuelto): {len(df_extra)}")
else:
    print("No se encontró textos_completos_verificados.csv, se omite esta fuente.")

print(f"Total candidatas SL a intentar (reiteraciones + research 5): {len(df_candidatas_sl)}")

Candidatas agregadas desde research 5 (texto completo ya resuelto): 2
Total candidatas SL a intentar (reiteraciones + research 5): 41


## 3. Resolución de PDF y descarga de texto

Cada página intermedia (`cortesuprema.gov.co/rl_dl_sl{id}`) tiene el PDF embebido en un
`<iframe src="...">`. La URL del PDF sigue el patrón
`.../relatorias/la/reiteraciones%20DL/{identificador}.pdf`, pero **no es seguro construirla
por patrón** — se comprobó un caso real (`SL3100-2024`) donde el identificador visible dice
2024 pero el archivo real es `SL3100-2023.pdf`. Por eso siempre se resuelve vía la página
intermedia, nunca por template.

El identificador se normaliza al mismo formato que ya usa el resto del corpus para sentencias
SL (`SL-3630/22`, de la parte 1): `SL{numero}-{año}` → `SL-{numero}/{año corto}`.

In [4]:
def normalizar_sl(identificador):
    m = re.match(r"SL(\d+)-(\d{4})", identificador)
    if not m:
        return identificador
    numero, anio = m.groups()
    return f"SL-{numero}/{anio[-2:]}"

RE_PDF = re.compile(r'(?:<iframe src|href)="([^"]+\.pdf)"')

def resolver_pdf_url(url_intermedia):
    # Entradas 2022+ traen el PDF en un <iframe src="...">; entradas 2017-2021 lo traen
    # en un <a href="..."> normal con el texto "Descargue el documento..." — hay que
    # cubrir ambos patrones, verificado contra ejemplos reales de cada época.
    try:
        resp = requests.get(url_intermedia, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        m = RE_PDF.search(resp.text)
        return m.group(1) if m else None
    except Exception:
        return None

def descargar_texto_pdf_sl(url_pdf):
    try:
        resp = requests.get(url_pdf, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        if resp.headers.get("content-type", "").split(";")[0] != "application/pdf":
            return None
        with pdfplumber.open(BytesIO(resp.content)) as pdf:
            texto = " ".join(page.extract_text() or "" for page in pdf.pages)
        return " ".join(texto.split())
    except Exception:
        return None

In [5]:
def descargar_texto_generico(url):
    # Igual que descargar_texto_pdf_sl, pero para candidatas con url_pdf_directo ya
    # verificada: acepta también HTML (SL592-2025 vive en normograma.crcom.gov.co como
    # página web, republicando el texto de la providencia, no un PDF).
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        content_type = resp.headers.get("content-type", "").split(";")[0]
        if content_type == "application/pdf":
            with pdfplumber.open(BytesIO(resp.content)) as pdf:
                texto = " ".join(page.extract_text() or "" for page in pdf.pages)
        elif "html" in content_type:
            soup = BeautifulSoup(resp.content, "html.parser")
            texto = soup.get_text(" ")
        else:
            return None
        return " ".join(texto.split())
    except Exception:
        return None

## 4. Prefiltro de keywords

Mismo prefiltro que los otros notebooks — aquí se espera que casi todas las candidatas lo
pasen, dado que ya vienen pre-filtradas por la pestaña "Laboral Individual" del sitio.

In [6]:
KEYWORDS_ALCANCE = [
    "estabilidad laboral reforzada", "contrato de trabajo", "despido",
    "terminación del contrato", "jornada laboral", "prestaciones sociales",
    "presunción de relación laboral", "liquidación", "recargo nocturno",
    "recargo dominical", "reintegro laboral",
    "relación laboral", "contrato realidad", "justa causa", "indemnización",
    "salario", "cesantías", "vacaciones", "ius variandi", "período de prueba",
    "discriminación laboral", "licencia de maternidad", "fuero de maternidad",
    "proceso disciplinario", "descuento salarial", "acoso laboral",
]

def pasa_prefiltro_keywords(texto):
    texto_lower = texto.lower()
    return any(kw in texto_lower for kw in KEYWORDS_ALCANCE)

## 5a. Filtro barato de pertinencia (Claude Haiku)

Mismo criterio corregido de los notebooks anteriores, con un ajuste nuevo: una EPS o AFP como
**empleadora demandada** (no como aseguradora en un trámite de salud/pensión) sí es laboral
individual válido — se agrega esa aclaración explícita para no perder casos reales por la sola
presencia de esas siglas en el texto.

Como estas sentencias son de Casación Laboral (no tutelas), no tienen sección `ANTECEDENTES`
en el mismo sentido — el fragmento arranca directo desde el inicio del texto.

In [7]:
PROMPT_GATE = """Lee este fragmento del inicio de una sentencia de la Sala de Casación Laboral de la Corte Suprema de Justicia de Colombia (puede estar incompleto). Decide si es relevante para un dataset de derecho laboral individual colombiano.

Devuelve SOLO un JSON válido, sin texto adicional ni backticks:
{{"pertinente_alcance": true o false, "razon": "una frase breve"}}

pertinente_alcance=true SOLO si el caso trata sobre una relación laboral INDIVIDUAL regida por el Código Sustantivo del Trabajo (o normas equivalentes para trabajadores oficiales): contratación, jornada, terminación del contrato, salario y pagos (constitutivos o no de salario, descuentos), licencias, vacaciones, cesantías (bajo régimen CST/Ley 50 de 1990), traslados (ius variandi), procedimiento disciplinario del empleador, estabilidad laboral reforzada, discriminación laboral en el empleo.

pertinente_alcance=false SIEMPRE, sin excepción, si:
- El tema central es PENSIÓN o SEGURIDAD SOCIAL en cualquier forma: pensión de vejez, invalidez, sobrevivientes, jubilación, sustitución pensional, bono pensional, régimen de prima media o ahorro individual. Esto aplica sin importar qué tan laboral suene el resto del caso — pensión siempre es false.
- El tema central es seguridad social EN SALUD (EPS): pago de licencias (maternidad, incapacidad) reclamado contra una EPS, cobertura o negación de servicios de salud, afiliación al sistema de salud.
- El accionante es un TRABAJADOR INDEPENDIENTE (por cuenta propia, sin empleador).
- El accionante es un ESTUDIANTE o PRACTICANTE en formación sin contrato de trabajo real.
- El empleador es una entidad estatal Y el vínculo es de "empleado público" bajo régimen estatutario/administrativo — NO CST. Excepción: "trabajador oficial" de empresa industrial/comercial del Estado sí puede ser pertinente.
- Es derecho colectivo (sindicatos, negociación colectiva, huelga, fuero sindical).
- Es función pública en cualquier otro sentido, o el tema es ajeno al laboral individual.
- Lo laboral aparece solo de forma incidental, sin ser el objeto central de la decisión.

IMPORTANTE — no confundas esto con lo anterior: si una EPS, AFP o Colpensiones aparece como la ENTIDAD DEMANDADA/EMPLEADORA en un pleito de salario, despido o prestaciones de uno de sus propios trabajadores, SÍ es pertinente — eso es laboral individual normal, no seguridad social. Solo es seguridad social cuando el litigio es sobre la prestación de salud/pensión en sí (afiliación, calificación, pago de la prestación), no cuando la EPS/AFP simplemente resulta ser el empleador del caso.

Si el fragmento no deja claro el tema o el tipo de vínculo laboral, responde false.

Fragmento:
{texto}
"""

def filtro_pertinencia_barato(texto, numero, chars_gate=3000):
    fragmento = texto[:chars_gate]
    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=200,
            messages=[{"role": "user", "content": PROMPT_GATE.format(texto=fragmento)}]
        )
        texto_respuesta = next(b.text for b in response.content if b.type == "text")
        raw = texto_respuesta.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"Error en filtro de pertinencia {numero}: {e}")
        return None

## 5b. Extracción estructurada completa (Claude Sonnet, solo pertinentes)

In [8]:
PROMPT_TEMPLATE = """Lee la siguiente sentencia laboral colombiana. Extrae exactamente esta información y devuelve SOLO un JSON válido, sin texto adicional ni backticks:

{{
  "hechos_resumidos": "los hechos del caso en 2-3 líneas, en lenguaje coloquial, como si un trabajador lo contara (ej. 'me despidieron después de...' o 'trabajé X años y...')",
  "pretension": "qué pedía el demandante, en pocas palabras",
  "articulos_fundamento_directo": ["artículos cuya interpretación/aplicación fue DETERMINANTE para resolver el punto concreto en disputa de este caso. Pregunta guía: si se quitara este artículo, ¿cambiaría el razonamiento de por qué se concedió o negó la pretensión específica? Si sí, va aquí. Formato 'CST Art. X' o 'Ley X de YYYY, Art. Y'"],
  "articulos_marco_general": ["artículos que la sentencia cita pero que son principios generales, reglas de interpretación/remisión, o contexto normativo (ej. favorabilidad, analogía, primacía de la realidad, normas constitucionales genéricas) — NO decidieron el punto específico del caso, cualquier sentencia laboral podría citarlos. Mismo formato."],
  "decision": "concedida" o "negada" o "parcial",
  "elemento_no_acreditado": "si la pretensión fue negada o parcial, qué elemento/requisito no se acreditó según el juez; si fue concedida totalmente, deja este campo vacío"
}}

Sentencia:
{texto}
"""

def extraer_estructura(texto, numero, max_chars=15000):
    texto_truncado = texto[:max_chars]
    try:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1200,
            messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(texto=texto_truncado)}]
        )
        texto_respuesta = next(b.text for b in response.content if b.type == "text")
        raw = texto_respuesta.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"Error extrayendo {numero}: {e}")
        return None

## 6. Construcción de pares (mismo criterio que los notebooks anteriores)

In [9]:
POOL_NEGATIVOS_FACILES = [
    {"articulo": "CST Art. 236", "tema": "licencia de maternidad"},
    {"articulo": "CST Art. 186", "tema": "vacaciones anuales"},
    {"articulo": "CST Art. 161", "tema": "jornada de trabajo"},
    {"articulo": "Ley 100 de 1993, Art. 13", "tema": "seguridad social"},
]

PROMPT_NEGATIVO_DIFICIL = """Dada esta consulta de un caso laboral colombiano:

{consulta}

Y sabiendo que los artículos correctamente aplicables son: {articulos_correctos}

Dame UN artículo real del derecho laboral colombiano (CST, leyes laborales) que esté relacionado temáticamente con la consulta pero que NO sea el fundamento correcto de la decisión — es decir, un artículo que alguien podría confundir con el correcto pero que no aplica aquí.

Devuelve SOLO un JSON válido, sin texto adicional ni backticks:
{{"articulo_incorrecto": "...", "por_que_se_confunde": "..."}}
"""

def generar_negativo_dificil(consulta, articulos_correctos):
    try:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=300,
            messages=[{"role": "user", "content": PROMPT_NEGATIVO_DIFICIL.format(
                consulta=consulta, articulos_correctos=articulos_correctos)}]
        )
        texto_respuesta = next(b.text for b in response.content if b.type == "text")
        raw = texto_respuesta.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"Error generando negativo difícil: {e}")
        return None

def construir_pares(extraccion, numero):
    consulta = extraccion["hechos_resumidos"]
    articulos_correctos = extraccion.get("articulos_fundamento_directo", [])
    articulos_marco = extraccion.get("articulos_marco_general", [])
    ya_citados = set(articulos_correctos) | set(articulos_marco)

    if not articulos_correctos:
        return None

    pares = [
        {"consulta": consulta, "articulo": art, "tipo": "positivo", "label": 1}
        for art in articulos_correctos
    ]

    candidatos_faciles = [n for n in POOL_NEGATIVOS_FACILES if n["articulo"] not in ya_citados]
    if candidatos_faciles:
        pares.append({
            "consulta": consulta,
            "articulo": candidatos_faciles[0]["articulo"],
            "tipo": "negativo_facil",
            "label": 0,
        })

    negativo_dificil = generar_negativo_dificil(consulta, articulos_correctos)
    if negativo_dificil:
        pares.append({
            "consulta": consulta,
            "articulo": negativo_dificil["articulo_incorrecto"],
            "tipo": "negativo_dificil_placeholder",
            "label": 0,
        })

    df = pd.DataFrame(pares)
    df["sentencia_origen"] = numero
    return df

## 7. Loop principal con checkpointing

Mismo patrón acumulado que los otros tres notebooks. Como el conjunto de candidatas es chico
(26), se procesan todas en una sola corrida — no hace falta un `SAMPLE_SIZE`.

In [10]:
CHECKPOINT_EVERY = 5

df_pares_acum = cargar_si_existe(PARES_PATH, COLUMNAS_PARES)
df_descartes_acum = cargar_si_existe(DESCARTES_PATH, COLUMNAS_DESCARTES)
ya_procesadas = set(df_pares_acum["sentencia_origen"]) | set(df_descartes_acum["sentencia"])
print(f"Ya procesadas (otros notebooks + corridas anteriores de este): {len(ya_procesadas)}")

df_candidatas_sl["sentencia_norm"] = df_candidatas_sl["identificador"].apply(normalizar_sl)
df_muestra = df_candidatas_sl[~df_candidatas_sl["sentencia_norm"].isin(ya_procesadas)].reset_index(drop=True)
print(f"Candidatas SL nuevas a intentar: {len(df_muestra)}")

nuevos_pares = []
nuevos_descartes = []

def guardar_checkpoint():
    partes = [df_pares_acum] + nuevos_pares
    pd.concat(partes, ignore_index=True).to_csv(PARES_PATH, index=False)
    pd.concat([df_descartes_acum, pd.DataFrame(nuevos_descartes, columns=COLUMNAS_DESCARTES)],
              ignore_index=True).to_csv(DESCARTES_PATH, index=False)

for i, row in df_muestra.iterrows():
    numero = row["sentencia_norm"]
    print(f"[{i+1}/{len(df_muestra)}] Procesando {row['identificador']} ({numero})...")

    try:
        url_pdf_directo = row.get("url_pdf_directo")
        if pd.notna(url_pdf_directo):
            texto = descargar_texto_generico(url_pdf_directo)
            time.sleep(0.5)
        else:
            url_pdf = resolver_pdf_url(row["url_intermedia"])
            time.sleep(0.5)

            if url_pdf is None:
                nuevos_descartes.append({"sentencia": numero, "razon": "no_se_encontro_iframe_pdf"})
                continue

            texto = descargar_texto_pdf_sl(url_pdf)
            time.sleep(0.5)

        if texto is None or len(texto) < 200:
            nuevos_descartes.append({"sentencia": numero, "razon": "sin_texto_o_pdf_no_resuelve"})
            continue

        if not pasa_prefiltro_keywords(texto):
            nuevos_descartes.append({"sentencia": numero, "razon": "no_pasa_prefiltro_keywords"})
            continue

        gate = filtro_pertinencia_barato(texto, numero)
        time.sleep(0.5)

        if gate is None:
            nuevos_descartes.append({"sentencia": numero, "razon": "error_filtro_pertinencia"})
            continue

        if not gate.get("pertinente_alcance"):
            nuevos_descartes.append({
                "sentencia": numero,
                "razon": f"llm_no_pertinente: {gate.get('razon', '')}",
            })
            continue

        extraccion = extraer_estructura(texto, numero)
        time.sleep(1)

        if extraccion is None:
            nuevos_descartes.append({"sentencia": numero, "razon": "error_extraccion_llm"})
            continue

        df_par = construir_pares(extraccion, numero)
        time.sleep(1)

        if df_par is not None:
            nuevos_pares.append(df_par)
        else:
            nuevos_descartes.append({"sentencia": numero, "razon": "sin_articulos_fundamento_directo"})

    except Exception as e:
        nuevos_descartes.append({"sentencia": numero, "razon": f"error_inesperado: {e}"})

    if (i + 1) % CHECKPOINT_EVERY == 0:
        guardar_checkpoint()
        print(f"  checkpoint guardado ({i+1}/{len(df_muestra)})")

guardar_checkpoint()
print(f"\nProcesamiento completo. Pares generados de {len(nuevos_pares)} sentencias pertinentes nuevas.")
print(f"Descartadas en esta corrida: {len(nuevos_descartes)} de {len(df_muestra)}")

Ya procesadas (otros notebooks + corridas anteriores de este): 1491
Candidatas SL nuevas a intentar: 2
[1/2] Procesando SL3186-2024 (SL-3186/24)...


[2/2] Procesando SL4965-2019 (SL-4965/19)...



Procesamiento completo. Pares generados de 2 sentencias pertinentes nuevas.
Descartadas en esta corrida: 0 de 2


## 8. Resultado acumulado

In [11]:
df_dataset_final = pd.concat([df_pares_acum] + nuevos_pares, ignore_index=True) if nuevos_pares else df_pares_acum
df_descartes_total = pd.concat([df_descartes_acum, pd.DataFrame(nuevos_descartes, columns=COLUMNAS_DESCARTES)], ignore_index=True)

print(f"--- Esta corrida ({len(df_muestra)} candidatas SL intentadas) ---")
print(f"Pares nuevos: {sum(len(df) for df in nuevos_pares)} de {len(nuevos_pares)} sentencias pertinentes")

print(f"\n--- Acumulado total (redal + embudo + research + SL) ---")
print(f"Total de pares: {len(df_dataset_final)}")
if not df_dataset_final.empty:
    print(f"Sentencias representadas: {df_dataset_final['sentencia_origen'].nunique()}")
    print(df_dataset_final["tipo"].value_counts())

print("\nRazones de descarte (acumulado):")
if not df_descartes_total.empty:
    print(df_descartes_total["razon"].apply(lambda r: r.split(":")[0]).value_counts())

print(f"\nGuardado en {PARES_PATH}")
print(f"Log de descartes en {DESCARTES_PATH}")
df_dataset_final.head()

--- Esta corrida (2 candidatas SL intentadas) ---
Pares nuevos: 10 de 2 sentencias pertinentes

--- Acumulado total (redal + embudo + research + SL) ---
Total de pares: 556
Sentencias representadas: 137
tipo
positivo                        282
negativo_facil                  137
negativo_dificil_placeholder    137
Name: count, dtype: int64

Razones de descarte (acumulado):
razon
llm_no_pertinente                   782
no_pasa_prefiltro_keywords          453
sin_articulos_fundamento_directo     50
sin_texto_o_url_no_resuelve          45
sin_texto_o_pdf_no_resuelve          14
revision_manual                      13
no_se_encontro_iframe_pdf             1
Name: count, dtype: int64

Guardado en ../../data/dataset_cross_encoder.csv
Log de descartes en ../../data/descartados.csv


,consulta,articulo,tipo,label,sentencia_origen
0,Trabajé como operador de bus articulado desde ...,CST Art. 127,positivo,1,SL-3630/22
1,Trabajé como operador de bus articulado desde ...,CST Art. 128,positivo,1,SL-3630/22
2,Trabajé como operador de bus articulado desde ...,CST Art. 236,negativo_facil,0,SL-3630/22
3,Trabajé como operador de bus articulado desde ...,CST Art. 130,negativo_dificil_placeholder,0,SL-3630/22
4,Trabajé para el ISS desde septiembre de 2000 h...,"Decreto 2127 de 1945, Art. 20",positivo,1,SL-2858/22
